# Lezione 34 — KerasHub: caricare un modello pre-addestrato

> **Modulo:** Transformer e modello open · **Tempo stimato:** 25 minuti
> **Prerequisiti:** Lezione 32 (blocco Transformer), Lezione 33 (generazione).
>
> Obiettivo pratico unico: capire **KerasHub** e l'API `from_preset` per
> caricare un modello pre-addestrato, distinguendo *backbone* e *task model*.
>
> ⚠️ **Nota di ambiente.** Questo notebook e' eseguibile in CI grazie a una
> guardia: le celle che scaricano/eseguono Gemma vengono saltate quando il
> pacchetto e i pesi non sono presenti (come in questo sandbox). Il codice
> mostrato e' l'API reale di KerasHub e gira su una macchina con GPU e accesso
> a Gemma configurato. Vedi `course/research_gaps.md`.

## Teoria essenziale

Riscrivere e riaddestrare un Transformer da zero (Lezioni 30–32) serve a
capirlo, non a usarlo in produzione. **KerasHub** e' la libreria che distribuisce
architetture e **preset** (pesi pre-addestrati) pronti. Due livelli:

- **Backbone**: la pila di blocchi Transformer, produce rappresentazioni.
- **Task model** (es. `GemmaCausalLM`): il backbone + la testa per un compito
  (qui: *causal language modeling*, cioe' generazione). Include il tokenizer.

Si carica tutto con un solo `from_preset("<nome-preset>")`.

In [1]:
# Guardia di ambiente: KerasHub + pesi Gemma sono un extra opzionale (`ml`)
# e richiedono download autenticato di un modello di diversi GB. In questo
# ambiente non sono presenti, quindi le celle che usano il modello vengono
# SALTATE in modo pulito (il notebook resta eseguibile in CI). Con GPU e
# accesso a Gemma configurato, la stessa cella gira davvero.
GEMMA_AVAILABLE = False
_motivo = ""
try:
    import keras  # noqa: F401
    import keras_hub  # noqa: F401
    GEMMA_AVAILABLE = True
except Exception as exc:  # noqa: BLE001
    _motivo = f"{type(exc).__name__}: {exc}"

if GEMMA_AVAILABLE:
    print("KerasHub disponibile: le celle del modello verranno eseguite.")
else:
    print("KerasHub/Gemma NON disponibile in questo ambiente.")
    print("Motivo:", _motivo or "pacchetto assente")
    print("Le celle che richiedono il modello stampano [saltato]; "
          "il resto della lezione gira comunque.")

KerasHub/Gemma NON disponibile in questo ambiente.
Motivo: ModuleNotFoundError: No module named 'keras'
Le celle che richiedono il modello stampano [saltato]; il resto della lezione gira comunque.


In [2]:
# API reale di KerasHub (eseguita solo se il modello e' disponibile).
if GEMMA_AVAILABLE:
    import keras_hub
    # scarica architettura + pesi + tokenizer del preset
    gemma = keras_hub.models.GemmaCausalLM.from_preset("gemma_2b_en")
    gemma.summary()
else:
    print("[saltato] keras_hub.models.GemmaCausalLM.from_preset('gemma_2b_en')")
    print("Questa chiamata scarica il preset (backbone + tokenizer + testa LM).")

[saltato] keras_hub.models.GemmaCausalLM.from_preset('gemma_2b_en')
Questa chiamata scarica il preset (backbone + tokenizer + testa LM).


## Il progetto: Memory AI Lab, passo 34

Non serve il modello per fissare il **contratto**: definisco un piccolo
registro dei preset che il progetto potrebbe usare, con i requisiti dichiarati.
E' la parte eseguibile e verificabile della lezione.

In [3]:
PRESET_REGISTRO = {
    "gemma_2b_en": {"parametri_mld": 2.0, "serve_gpu": True, "lingua": "en"},
    "gemma_2b_multi": {"parametri_mld": 2.0, "serve_gpu": True, "lingua": "multi"},
}

def scegli_preset(serve_multilingua: bool) -> str:
    for nome, meta in PRESET_REGISTRO.items():
        if meta["lingua"] == ("multi" if serve_multilingua else "en"):
            return nome
    raise ValueError("nessun preset adatto")

# controllo di non-regressione (gira sempre, senza modello)
assert scegli_preset(False) == "gemma_2b_en"
assert PRESET_REGISTRO["gemma_2b_en"]["serve_gpu"] is True
print("preset per progetto EN:", scegli_preset(False))
print("preset per progetto multilingua:", scegli_preset(True))

preset per progetto EN: gemma_2b_en
preset per progetto multilingua: gemma_2b_multi


## Riepilogo

1. KerasHub distribuisce architetture e **preset** (pesi pre-addestrati).
2. `from_preset("...")` carica backbone + tokenizer + testa in un colpo.
3. **Backbone** = rappresentazioni; **task model** = backbone + testa (es.
   `GemmaCausalLM` per la generazione).
4. Un preset da 2 mld di parametri richiede tipicamente una GPU.
5. Il contratto (quale preset, quali requisiti) si puo' fissare e testare anche
   senza il modello caricato.

## Quiz

1. Che differenza c'e' tra backbone e task model?
2. Perche' `from_preset` scarica anche il tokenizer?
3. Cosa cambia tra addestrare da zero (Lezione 32) e usare un preset?

*(Risposte: 1. il backbone produce rappresentazioni, il task model aggiunge la
testa per un compito specifico; 2. perche' il modello e' stato addestrato con
quel tokenizer esatto e gli id devono corrispondere; 3. il preset porta pesi
gia' appresi su enormi corpora, si evita l'addestramento costoso.)*